In [ ]:
from google.colab import files
uploaded = files.upload()

Saving collegiate_athlete_injury_dataset.csv to collegiate_athlete_injury_dataset.csv


In [ ]:
import pandas as pd

df = pd.read_csv(list(uploaded.keys())[0])
df.head()

,Athlete_ID,Age,Gender,Height_cm,Weight_kg,Position,Training_Intensity,Training_Hours_Per_Week,Recovery_Days_Per_Week,Match_Count_Per_Week,Rest_Between_Events_Days,Fatigue_Score,Performance_Score,Team_Contribution_Score,Load_Balance_Score,ACL_Risk_Score,Injury_Indicator
0,A001,24,Female,195,99,Center,2,13,2,3,1,1,99,58,100,4,0
1,A002,21,Male,192,65,Forward,8,14,1,3,1,4,55,63,83,73,0
2,A003,22,Male,163,83,Guard,8,8,2,1,3,6,58,62,100,62,0
3,A004,24,Female,192,90,Guard,1,13,1,1,1,7,82,74,78,51,0
4,A005,20,Female,173,79,Center,3,9,1,2,1,2,90,51,83,49,0


In [ ]:
# Drop ID column
if 'Athlete_ID' in df.columns:
    df = df.drop('Athlete_ID', axis=1)

# Encode categorical
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

In [ ]:
X = df.drop(['Performance_Score', 'Injury_Indicator'], axis=1)
y = df['Injury_Indicator']

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

svm = SVC(kernel='rbf', class_weight='balanced')
svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=1))

SVM Accuracy: 0.9
              precision    recall  f1-score   support

           0       0.92      0.97      0.95        37
           1       0.00      0.00      0.00         3

    accuracy                           0.90        40
   macro avg       0.46      0.49      0.47        40
weighted avg       0.85      0.90      0.88        40



In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

grid = GridSearchCV(SVC(), params, cv=3)
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

Best Parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}


In [ ]:
best_svm = grid.best_estimator_

y_pred = best_svm.predict(X_test)

print("Optimized Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=1))

Optimized Accuracy: 0.925
              precision    recall  f1-score   support

           0       0.93      1.00      0.96        37
           1       1.00      0.00      0.00         3

    accuracy                           0.93        40
   macro avg       0.96      0.50      0.48        40
weighted avg       0.93      0.93      0.89        40



In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[37  0]
 [ 3  0]]
